In [13]:

from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

np.random.seed(42)
tf.random.set_seed(42)

df = pd.read_csv("bengaluru_house_prices.csv")

df = df.drop_duplicates()

def convert_sqft(x):
    try:
        x = str(x).strip()

        if "-" in x:
            a, b = x.split("-")
            return (float(a) + float(b)) / 2

        return float(x)

    except:
        return np.nan

df["total_sqft"] = df["total_sqft"].apply(convert_sqft)

df["bhk"] = (
    df["size"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
)

df["bhk"] = pd.to_numeric(
    df["bhk"],
    errors="coerce"
)

for col in ["bath", "balcony", "price"]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df = df.drop(
    columns=["size", "society"],
    errors="ignore"
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna(
    subset=[
        "location",
        "total_sqft",
        "bhk",
        "bath",
        "price"
    ]
)

df = df[
    (df["total_sqft"] >= 300) &
    (df["total_sqft"] <= 10000) &
    (df["bhk"] >= 1) &
    (df["bhk"] <= 20) &
    (df["bath"] >= 1) &
    (df["price"] > 0)
]

df["sqft_per_bhk"] = (
    df["total_sqft"] / df["bhk"]
)

df["bath_per_bhk"] = (
    df["bath"] / df["bhk"]
)

df["bath_bhk_ratio"] = (
    df["bath"] / df["bhk"]
)

df["sqft_log"] = np.log1p(
    df["total_sqft"]
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna()

train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42
)

global_mean = train_df["price"].mean()

location_stats = (
    train_df
    .groupby("location")["price"]
    .agg(["mean", "count"])
)

smooth = 20

location_encoding = (
    (
        location_stats["mean"] *
        location_stats["count"]
    )
    +
    (
        global_mean *
        smooth
    )
) / (
    location_stats["count"] +
    smooth
)

train_df["location_encoded"] = (
    train_df["location"]
    .map(location_encoding)
    .fillna(global_mean)
)

test_df["location_encoded"] = (
    test_df["location"]
    .map(location_encoding)
    .fillna(global_mean)
)

train_df = train_df.drop(
    columns=[
        "location",
        "availability",
        "area_type"
    ],
    errors="ignore"
)

test_df = test_df.drop(
    columns=[
        "location",
        "availability",
        "area_type"
    ],
    errors="ignore"
)

train_df = train_df.replace(
    [np.inf, -np.inf],
    np.nan
)

test_df = test_df.replace(
    [np.inf, -np.inf],
    np.nan
)

train_df = train_df.dropna()
test_df = test_df.dropna()

X_train = train_df.drop(
    columns=["price"]
)

y_train = np.log1p(
    train_df["price"]
)

X_test = test_df.drop(
    columns=["price"]
)

y_test = np.log1p(
    test_df["price"]
)

X_train = pd.get_dummies(
    X_train,
    dtype=float
)

X_test = pd.get_dummies(
    X_test,
    dtype=float
)

X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)

X_train = X_train.fillna(
    X_train.median()
)

X_test = X_test.fillna(
    X_train.median()
)

X_train = X_train.astype(
    np.float32
)

X_test = X_test.astype(
    np.float32
)

y_train = y_train.astype(
    np.float32
)

y_test = y_test.astype(
    np.float32
)

print("NaN in X_train:", np.isnan(X_train).sum())
print("NaN in X_test :", np.isnan(X_test).sum())
print("NaN in y_train:", np.isnan(y_train).sum())
print("NaN in y_test :", np.isnan(y_test).sum())

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_test = scaler.transform(
    X_test
)

X_train = np.nan_to_num(
    X_train,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X_test = np.nan_to_num(
    X_test,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

model = Sequential([
    Input(
        shape=(X_train.shape[1],)
    ),

    Dense(
        128,
        activation="relu"
    ),

    BatchNormalization(),

    Dropout(0.10),

    Dense(
        64,
        activation="relu"
    ),

    BatchNormalization(),

    Dropout(0.10),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1
    )
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="mse",
    metrics=["mae"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.15,
    epochs=150,
    batch_size=64,
    callbacks=[
        early_stop,
        reduce_lr
    ],
    verbose=1
)

pred_log = model.predict(
    X_test,
    verbose=0
).flatten()

pred = np.expm1(
    pred_log
)

actual = np.expm1(
    y_test
)

valid = (
    np.isfinite(actual) &
    np.isfinite(pred)
)

actual = actual[valid]
pred = pred[valid]

r2 = r2_score(
    actual,
    pred
)

mae = mean_absolute_error(
    actual,
    pred
)

mape = np.mean(
    np.abs(actual - pred) /
    np.maximum(actual, 1e-8)
) * 100

accuracy_10 = np.mean(
    np.abs(actual - pred) /
    actual <= 0.10
) * 100

accuracy_20 = np.mean(
    np.abs(actual - pred) /
    actual <= 0.20
) * 100

accuracy_30 = np.mean(
    np.abs(actual - pred) /
    actual <= 0.30
) * 100

accuracy_50 = np.mean(
    np.abs(actual - pred) /
    actual <= 0.50
) * 100

print()
print("==============================")
print("       MODEL RESULTS")
print("==============================")
print(f"R2 Score          : {r2:.4f}")
print(f"R2 Percentage     : {r2 * 100:.2f}%")
print(f"MAE               : {mae:.2f}")
print(f"MAPE              : {mape:.2f}%")
print(f"Accuracy ±10%     : {accuracy_10:.2f}%")
print(f"Accuracy ±20%     : {accuracy_20:.2f}%")
print(f"Accuracy ±30%     : {accuracy_30:.2f}%")
print(f"Accuracy ±50%     : {accuracy_50:.2f}%")
print("==============================")

Saving bengaluru_house_prices.csv to bengaluru_house_prices (5).csv
NaN in X_train: total_sqft          0
bath                0
balcony             0
bhk                 0
sqft_per_bhk        0
bath_per_bhk        0
bath_bhk_ratio      0
sqft_log            0
location_encoded    0
dtype: int64
NaN in X_test : total_sqft          0
bath                0
balcony             0
bhk                 0
sqft_per_bhk        0
bath_per_bhk        0
bath_bhk_ratio      0
sqft_log            0
location_encoded    0
dtype: int64
NaN in y_train: 0
NaN in y_test : 0
Epoch 1/150
137/137 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 3.0929 - mae: 1.3129 - val_loss: 5.5864 - val_mae: 2.3182 - learning_rate: 5.0000e-04
Epoch 2/150
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4962 - mae: 0.5412 - val_loss: 1.0809 - val_mae: 0.9521 - learning_rate: 5.0000e-04
Epoch 3/150
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3892 - mae: 0.4817 - val_loss: 0.2164 - val_mae: 0.3421 - learning_rate: 5.0000e-04
E